# Lab 10 — End-to-End Task Orchestration with Databricks Jobs & Unity Catalog

**Goal:** Review data governance grants and prepare a multi-task workflow.

**Important:** Job creation and notification setup are performed in the Databricks Jobs UI or via the Jobs API/CLI. This notebook provides runnable permission examples and a JSON job-spec template. Replace all placeholder paths and pipeline IDs before deployment.

In [0]:
%sql
-- Inspect current catalog and schema
SELECT current_catalog() AS current_catalog, current_schema() AS current_schema;

## Unity Catalog grants

These commands require an owner/admin or a principal with grant permissions. The principal `data_analysts` must already exist as a group. Run the statements one at a time and adjust catalog/schema/table names to your environment.

In [0]:
%sql
-- GRANT USE CATALOG ON CATALOG workspace TO `data_analysts`;
-- GRANT USE SCHEMA ON SCHEMA workspace.lab_db TO `data_analysts`;
-- GRANT SELECT ON TABLE workspace.lab_db.gold_daily_revenue_lab8 TO `data_analysts`;

In [0]:
%sql
-- Verify table availability before wiring the job
SHOW TABLES IN workspace.lab_db;

## Multi-task job JSON template

Save the JSON below as `lab10_job.json` and replace the notebook path and pipeline ID. Import/create the job through the Databricks Jobs UI or CLI. Task keys and paths must match resources in your workspace.

In [0]:
job_spec = {
  "name": "End-To-End Medallion Lakehouse Job - Lab 10",
  "tasks": [
    {
      "task_key": "Ingest_AutoLoader",
      "notebook_task": {
        "notebook_path": "/Users/<your-user>/Lab_06_Auto_Loader_Ingestion",
        "source": "WORKSPACE"
      },
      "max_retries": 1,
      "min_retry_interval_millis": 60000,
      "timeout_seconds": 1800
    },
    {
      "task_key": "Run_Silver_Cleansing",
      "depends_on": [{"task_key": "Ingest_AutoLoader"}],
      "notebook_task": {
        "notebook_path": "/Users/<your-user>/Lab_07_Bronze_to_Silver_Cleansing",
        "source": "WORKSPACE"
      },
      "max_retries": 1,
      "timeout_seconds": 1800
    }
  ],
  "format": "MULTI_TASK"
}

import json
print(json.dumps(job_spec, indent=2))

## Final checklist

- [ ] Confirm the `data_analysts` group exists before running GRANT commands.
- [ ] Confirm the target Gold table exists (run Lab 8 first).
- [ ] Replace notebook paths with the actual paths in your workspace.
- [ ] Configure a job cluster or compatible compute for each task.
- [ ] Add failure notifications in the Job settings UI.
- [ ] Run the job once and inspect each task's output and run history.

**Lab series complete:** Labs 1–10 cover SQL, Delta fundamentals, streaming ingestion, medallion modeling, pipeline development, orchestration, and governance.